**Task 2 (0.65p).** WordNet mini-challenge

(0.25p) Exploit WordNet’s hierarchy using relations such as hyponym/hypernym, antonyms,
meronyms, synonyms and definitions to collect words that are related to a user given word.

(0.25p) Build a Word Association Game using WordNet Similarity (any existing similarity method
from NLTK library), along with the relations identified above. The player is prompted with a word
and must come up with related words based on semantic similarity and relations using WordNet.
Offer rewards points based on how semantically similar the player's word is to the original word.
Provide feedback on the "closeness" of the player's word to the original.

(0.15p) Provide it in a user-friendly manner.

In [24]:
x = int(input("give me a number"))
print(x+1)

2


In [25]:
from nltk.corpus import wordnet as wn

In [26]:
def clean_word(word: str) -> str:
    return word.lower().replace('_', ' ')

In [27]:
def get_related_words(word: str) -> None | dict[str, set[str] | list[str]]:
    """
    Collects related words from WordNet across multiple semantic relations:
    - Synonyms
    - Antonyms
    - Hypernyms (parents)
    - Hyponyms (children)
    - Meronyms (parts, substances, members)
    """
    synsets = wn.synsets(word)
    if len(synsets) == 0:
        return None

    target_clean = clean_word(word)

    results = {
        "definitions": [],
        "synonyms": set(),
        "antonyms": set(),
        "hypernyms": set(),
        "hyponyms": set(),
        "meronyms": set()
    }

    for synset in synsets:
        results["definitions"].append(f"({synset.pos()}) {synset.definition()}")

        # Synonyms and Antonyms (Lemma-level relations)
        for lemma in synset.lemmas():
            synonym_clean = clean_word(lemma.name())
            if synonym_clean != target_clean:
                results["synonyms"].add(synonym_clean)
            
            for antonym in lemma.antonyms():
                results["antonyms"].add(clean_word(antonym.name()))

        # Hypernyms (Broader terms)
        for hypernym_synset in synset.hypernyms():
            for lemma in hypernym_synset.lemmas():
                results["hypernyms"].add(clean_word(lemma.name()))

        # Hyponyms (More specific terms)
        for hyponym_synset in synset.hyponyms():
            for lemma in hyponym_synset.lemmas():
                results["hyponyms"].add(clean_word(lemma.name()))

        # Meronyms
        meronym_synsets = synset.part_meronyms() + synset.substance_meronyms() + synset.member_meronyms()
        for meronym_synset in meronym_synsets:
            for lemma in meronym_synset.lemmas():
                results["meronyms"].add(clean_word(lemma.name()))

    # Remove the target word itself from all sets
    for key in ["synonyms", "antonyms", "hypernyms", "hyponyms", "meronyms"]:
        results[key].discard(target_clean)

    return results

In [28]:
from pprint import pprint
pprint(get_related_words("car"))

{'antonyms': set(),
 'definitions': ['(n) a motor vehicle with four wheels; usually propelled by '
                 'an internal combustion engine',
                 '(n) a wheeled vehicle adapted to the rails of railroad',
                 '(n) the compartment that is suspended from an airship and '
                 'that carries personnel and the cargo and the power plant',
                 '(n) where passengers ride up and down',
                 '(n) a conveyance for passengers or freight on a cable '
                 'railway'],
 'hypernyms': {'automotive vehicle',
               'compartment',
               'motor vehicle',
               'wheeled vehicle'},
 'hyponyms': {'ambulance',
              'baggage car',
              'beach waggon',
              'beach wagon',
              'bus',
              'cab',
              'cabin car',
              'caboose',
              'carriage',
              'club car',
              'coach',
              'compact',
              'co

In [29]:
# =====================================================================
# PART 2: Semantic Similarity & Relation Evaluator
# =====================================================================

def evaluate_association(target_word: str, guess_word: str) -> tuple[float, str, list]:
    """
    Evaluates how closely target_word and guess_word are related using:
    1. WordNet Path Similarity (Semantic closeness)
    2. Explicit WordNet structural relations
    """
    target_synsets = wn.synsets(target_word)
    guess_synsets = wn.synsets(guess_word)

    if len(target_synsets) == 0 or len(guess_synsets) == 0:
        return 0.0, "Word not found in WordNet dictionary.", []

    # Calculate highest path similarity score across synset pairs
    max_similarity = 0.0
    for target_synset in target_synsets:
        for guess_synset in guess_synsets:
            # path_similarity returns a float [0, 1] or None
            path_similarity: float | None = target_synset.path_similarity(guess_synset)
            if path_similarity is not None and path_similarity > max_similarity:
                max_similarity = path_similarity

    # Identify direct structural relations
    relations_found: list[str] = []
    rel_data = get_related_words(target_word)
    clean_guess = clean_word(guess_word)

    if rel_data:
        for rel_type in ["synonyms", "antonyms", "hypernyms", "hyponyms", "meronyms"]:
            if clean_guess in rel_data[rel_type]:
                relations_found.append(rel_type.capitalize())

    # Formulate feedback based on similarity score
    if max_similarity >= 0.5:
        feedback = "🔥 Exceptional! Highly semantically related."
    elif max_similarity >= 0.33:
        feedback = "✨ Great match! Strong relationship."
    elif max_similarity >= 0.20:
        feedback = "👍 Decent connection, moderate similarity."
    elif max_similarity >= 0.10:
        feedback = "❄️ Cold... A weak semantic link."
    else:
        feedback = "🥶 Very Distant. Hardly any connection found."

    return max_similarity, feedback, relations_found

In [ ]:
# =====================================================================
# PART 3: Terminal UI
# =====================================================================

import random
from IPython.display import clear_output
from IPython.terminal import prompts

# Color code constants for terminal styling
class Colors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    DIM = '\033[2m'

WORD_POOL = [
    "ocean", "dog", "car", "computer", "music", 
    "forest", "book", "fire", "house", "light",
    "planet", "king", "apple", "doctor", "mountain"
]

def clear_screen():
    """Clears the terminal screen across different operating systems."""
    clear_output()

def print_banner():
    """Prints a styled game header."""
    print(f"{Colors.OKCYAN}{'='*60}")
    print(f"{'🎮  WORDNET SEMANTIC ASSOCIATION GAME  🎮':^60}")
    print(f"{'='*60}{Colors.ENDC}\n")

def run_game():
    total_score = 0
    rounds_played = 0
    current_word = random.choice(WORD_POOL)
    last_feedback = "Enter a word related to the target!"
    last_points = 0
    last_relations = []
    last_guess = ""

    clear_screen()

    while True:
        print_banner()

        # Score & Status Bar
        print(f" {Colors.BOLD}Total Score:{Colors.ENDC} {Colors.OKGREEN}{total_score} pts{Colors.ENDC}  |  {Colors.BOLD}Rounds Played:{Colors.ENDC} {rounds_played}")
        print(f"{Colors.DIM}{'-'*60}{Colors.ENDC}\n")

        # Target Word & Definition
        print(f" 🎯 {Colors.BOLD}Target Word:{Colors.ENDC} {Colors.WARNING}{current_word.upper()}{Colors.ENDC}")
        
        rel_info = get_related_words(current_word)
        if rel_info and rel_info["definitions"]:
            def_text = rel_info["definitions"][0]
            if len(def_text) > 55:
                def_text = def_text[:52] + "..."
            print(f"    {Colors.DIM}Def: {def_text}{Colors.ENDC}")
        print()

        # Last Round Results
        if last_guess:
            print(f"{Colors.HEADER}┌── Last Round Results ──────────────────────────────────┐{Colors.ENDC}")
            print(f"  Your Word: '{Colors.BOLD}{last_guess}{Colors.ENDC}' -> {last_feedback}")
            rel_str = ', '.join(last_relations) if last_relations else "None"
            print(f"  Earned: {Colors.OKGREEN}+{last_points} pts{Colors.ENDC}  |  Direct Relations: {Colors.HEADER}{rel_str}{Colors.ENDC}")
            print(f"{Colors.HEADER}└────────────────────────────────────────────────────────┘{Colors.ENDC}\n")

        # Input Prompt
        print(f"Type {Colors.WARNING}'pass'{Colors.ENDC} for a new word, or {Colors.FAIL}'quit'{Colors.ENDC} to exit.")
        try:
            guess = input(f"{Colors.BOLD}Enter your related word ❯ {Colors.ENDC}").strip().lower()
        except (KeyboardInterrupt, EOFError):
            print("\nExiting game...")
            break

        if not guess:
            clear_screen()
            continue

        if guess == 'quit':
            print(f"\n{Colors.OKGREEN}Thanks for playing! Final Score: {total_score} pts{Colors.ENDC}")
            break

        if guess == 'pass':
            current_word = random.choice([w for w in WORD_POOL if w != current_word])
            last_guess = ""
            last_feedback = "Word skipped! New target generated."
            clear_screen()
            continue

        # Evaluate Guess
        sim, feedback, relations = evaluate_association(current_word, guess)

        if feedback == "Word not found in WordNet dictionary.":
            last_guess = guess
            last_feedback = f"{Colors.FAIL}❌ Word not recognized in WordNet.{Colors.ENDC}"
            last_points = 0
            last_relations = []
            clear_screen()
            continue

        # Calculate Points
        base_points = int(sim * 100)
        bonus = len(relations) * 20
        last_points = base_points + bonus

        total_score += last_points
        rounds_played += 1
        last_guess = guess
        last_feedback = feedback
        last_relations = relations

        # Next Target Word
        current_word = random.choice([w for w in WORD_POOL if w != current_word])
        clear_screen()

In [31]:
run_game()

          🎮  WORDNET SEMANTIC ASSOCIATION GAME  🎮           

 Total Score: 0 pts  |  Rounds Played: 0
------------------------------------------------------------

 🎯 Target Word: MUSIC
    Def: (n) an artistic form of auditory communication incor...

Type 'pass' for a new word, or 'quit' to exit.


AttributeError: module 'IPython.terminal.prompts' has no attribute 'input'